# Ensure lakeLoom Service Principals

Creates (or finds) two least-privilege service principals for the lakeLoom platform:

1. **ZeroBus SPN** (`lakeloom-{schema}`) — dedicated to streaming data from the
   AppKit server to the bronze table via the ZeroBus SDK. Scoped to a single
   schema and secret scope.
2. **Xcode SPN** (`lakeloom-xcode-{schema}`) — dedicated to authenticating the
   iOS/iPadOS/macOS app to the Databricks App API endpoints before QR-pair
   onboarding completes. Granted minimal permissions (App API access only).

This notebook runs as the **first task** in the `platform_bootstrap` job. It
outputs the ZeroBus SPN `application_id` as a **task value** so the downstream
DDL task can grant table-level permissions.

**Secret scope keys auto-provisioned by this notebook:**

| Key | Source | Purpose |
| --- | --- | --- |
| `{client_id_dbs_key}` | ZeroBus SPN `application_id` | OAuth M2M client identifier |
| `{xcode_client_id_dbs_key}` | Xcode SPN `application_id` | iOS app OAuth M2M client identifier |
| `workspace_url` | Derived from config | Databricks workspace URL |
| `zerobus_endpoint` | Workspace ID + region | ZeroBus gRPC endpoint |
| `target_table_name` | Catalog + schema | Fully qualified bronze table |
| `zerobus_stream_pool_size` | Job parameter | SDK gRPC stream pool size |

**Admin-provisioned (manual after first deploy):**

| Key | Source | Purpose |
| --- | --- | --- |
| `{client_secret_dbs_key}` | Workspace UI or CLI | ZeroBus OAuth M2M client secret |
| `{xcode_client_secret_dbs_key}` | Workspace UI or CLI | Xcode OAuth M2M client secret |

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog_use", "")
dbutils.widgets.text("schema_use", "")
dbutils.widgets.text("secret_scope_name", "")
dbutils.widgets.text("client_id_dbs_key", "")
dbutils.widgets.text("client_secret_dbs_key", "")
dbutils.widgets.text("xcode_client_id_dbs_key", "")
dbutils.widgets.text("xcode_client_secret_dbs_key", "")
dbutils.widgets.text("zerobus_stream_pool_size", "")

catalog_use = dbutils.widgets.get("catalog_use")
schema_use = dbutils.widgets.get("schema_use")
secret_scope_name = dbutils.widgets.get("secret_scope_name")
client_id_dbs_key = dbutils.widgets.get("client_id_dbs_key")
client_secret_dbs_key = dbutils.widgets.get("client_secret_dbs_key")
xcode_client_id_dbs_key = dbutils.widgets.get("xcode_client_id_dbs_key")
xcode_client_secret_dbs_key = dbutils.widgets.get("xcode_client_secret_dbs_key")
zerobus_stream_pool_size = dbutils.widgets.get("zerobus_stream_pool_size")

print(f"Catalog:              {catalog_use}")
print(f"Schema:               {schema_use}")
print(f"Secret scope:         {secret_scope_name}")
print(f"Client ID key:        {client_id_dbs_key}")
print(f"Client secret key:    {client_secret_dbs_key}")
print(f"Xcode ID key:         {xcode_client_id_dbs_key}")
print(f"Xcode secret key:     {xcode_client_secret_dbs_key}")
print(f"Stream pool size:     {zerobus_stream_pool_size}")

In [0]:
import sys
from pathlib import Path

# Add src/lib to the import path (relative to this notebook)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
lib_path = str(Path("/Workspace") / Path(notebook_path).parent.relative_to("/") / ".." / "lib")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from workspace_metadata import get_workspace_id, get_region, get_zerobus_endpoint
from service_principal import get_or_create_service_principal, verify_client_credentials
from secret_scope import put_secret, list_secret_keys, try_get_secret_value

print(f"Loaded lib from: {lib_path}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
workspace_url = w.config.host.rstrip("/")

workspace_id = get_workspace_id(w)
if not workspace_id:
    raise RuntimeError(
        "Could not determine workspace ID. "
        "Set DATABRICKS_WORKSPACE_ID or run on supported Databricks compute."
    )

region = get_region(workspace_url)
zerobus_endpoint = get_zerobus_endpoint(workspace_id, region)
target_table_name = f"{catalog_use}.{schema_use}.transcript_events_raw"

print(f"Workspace URL:        {workspace_url}")
print(f"Workspace ID:         {workspace_id}")
print(f"AWS region:           {region}")
print(f"ZeroBus endpoint:     {zerobus_endpoint}")
print(f"Target table:         {target_table_name}")

In [0]:
spn_display_name = f"lakeloom-{schema_use}"
print(f"Target SPN display name: {spn_display_name}")

spn, is_new_spn = get_or_create_service_principal(w, spn_display_name)
spn_application_id = spn.application_id

print(f"Application ID:       {spn_application_id}")
print(f"Workspace object ID:  {spn.id}")
print(f"Created this run:     {is_new_spn}")

In [0]:
# ---------------------------------------------------------------------------
# Find or Create the Xcode (Apple Platforms) Service Principal
#
# The Xcode SPN authenticates the iOS/iPadOS/macOS app to the Databricks
# App API endpoints. It has NO data-plane permissions — the App's route
# guard identifies it from proxy-injected headers and restricts access to
# pairing and session endpoints only.
#
# Naming convention: lakeloom-xcode-{schema}
# ---------------------------------------------------------------------------

xcode_spn_display_name = f"lakeloom-xcode-{schema_use}"
print(f"Target Xcode SPN display name: {xcode_spn_display_name}")

xcode_spn, is_new_xcode_spn = get_or_create_service_principal(w, xcode_spn_display_name)
xcode_spn_application_id = xcode_spn.application_id

print(f"Application ID:       {xcode_spn_application_id}")
print(f"Workspace object ID:  {xcode_spn.id}")
print(f"Created this run:     {is_new_xcode_spn}")

In [0]:
secrets_to_store = {
    client_id_dbs_key: spn_application_id,
    xcode_client_id_dbs_key: xcode_spn_application_id,
    "workspace_url": workspace_url,
    "zerobus_endpoint": zerobus_endpoint,
    "target_table_name": target_table_name,
    "zerobus_stream_pool_size": zerobus_stream_pool_size,
}

for key, value in secrets_to_store.items():
    put_secret(w, secret_scope_name, key, value)
    print(f"Stored {key} = {value}")

# Check for admin-provisioned secrets
existing_keys = list_secret_keys(w, secret_scope_name)
credentials_provisioned = client_secret_dbs_key in existing_keys
xcode_credentials_provisioned = xcode_client_secret_dbs_key in existing_keys
print(f"\nAvailable keys:            {sorted(existing_keys)}")
print(f"ZeroBus secret present:    {'YES' if credentials_provisioned else 'NO — admin action required'}")
print(f"Xcode secret present:      {'YES' if xcode_credentials_provisioned else 'NO — admin action required'}")

In [0]:
# ---------------------------------------------------------------------------
# Secret Scope READ Access — NOT granted to these SPNs
#
# Neither the ZeroBus SPN nor the Xcode SPN needs READ on the secret scope:
#   - ZeroBus SPN only authenticates to OIDC and writes to the bronze table
#   - Xcode SPN only authenticates to the Databricks App's API endpoints
#
# The Databricks App's auto-provisioned SPN (created when the App deploys)
# is the principal that needs READ — it reads client_id, client_secret,
# workspace_url, zerobus_endpoint, etc. to configure itself at runtime.
#
# Grant App SPN READ via: src/admin_actions/update-secrets-acls notebook
# or the companion lakeLoom_app bundle's bootstrap.
# ---------------------------------------------------------------------------

print("Secret scope READ: NOT granted to ZeroBus or Xcode SPNs (by design)")
print("  → ZeroBus SPN only needs: USE CATALOG, USE SCHEMA, MODIFY+SELECT on target table")
print("  → Xcode SPN only needs: CAN_USE on the Databricks App resource")
print("  → App SPN (auto-provisioned at App deploy) gets READ on the scope separately")

In [0]:
m2m_token_verified = False
m2m_verification_status = "skipped"
verification_details = "client_secret not available to this run"

if credentials_provisioned:
    client_secret_value, read_error = try_get_secret_value(
        secret_scope_name, client_secret_dbs_key
    )
    if client_secret_value:
        ok, status_code, preview = verify_client_credentials(
            workspace_url=workspace_url,
            client_id=spn_application_id,
            client_secret=client_secret_value,
        )
        m2m_token_verified = ok
        m2m_verification_status = f"http_{status_code}"
        verification_details = preview or "token response had empty body"
        if not ok:
            raise RuntimeError(
                f"OAuth client credentials verification failed "
                f"with status {status_code}: {preview}"
            )
        print(f"M2M token verification: PASSED (HTTP {status_code})")
    else:
        m2m_verification_status = "skipped"
        verification_details = (
            f"client_secret exists but could not be read: {read_error}"
        )
        print(f"M2M token verification: SKIPPED — {verification_details}")
else:
    print("M2M token verification: SKIPPED — client_secret not yet provisioned")

In [0]:
# The DDL task reads the ZeroBus SPN via:
#   {{tasks.ensure_service_principal.values.spn_application_id}}
dbutils.jobs.taskValues.set(key="spn_application_id", value=spn_application_id)
dbutils.jobs.taskValues.set(key="xcode_spn_application_id", value=xcode_spn_application_id)
print(f"Task value set: spn_application_id       = {spn_application_id}")
print(f"Task value set: xcode_spn_application_id = {xcode_spn_application_id}")

In [0]:
import json

summary = {
    "zerobus_spn": {
        "display_name": spn_display_name,
        "application_id": spn_application_id,
        "workspace_object_id": spn.id,
        "created_this_run": is_new_spn,
        "client_secret_present": credentials_provisioned,
        "m2m_token_verified": m2m_token_verified,
        "m2m_verification_status": m2m_verification_status,
    },
    "xcode_spn": {
        "display_name": xcode_spn_display_name,
        "application_id": xcode_spn_application_id,
        "workspace_object_id": xcode_spn.id,
        "created_this_run": is_new_xcode_spn,
        "client_secret_present": xcode_credentials_provisioned,
    },
    "workspace_url": workspace_url,
    "workspace_id": workspace_id,
    "aws_region": region,
    "zerobus_endpoint": zerobus_endpoint,
    "target_table_name": target_table_name,
    "secret_scope_name": secret_scope_name,
    "next_manual_steps": [
        f"Generate and store {client_secret_dbs_key} in scope {secret_scope_name} for ZeroBus SPN {spn_application_id}",
        f"Generate and store {xcode_client_secret_dbs_key} in scope {secret_scope_name} for Xcode SPN {xcode_spn_application_id}",
    ],
}

print("=" * 70)
print("Platform Bootstrap — Service Principals Summary")
print("=" * 70)
print(json.dumps(summary, indent=2, sort_keys=True))